# Stage 2: Mandelbugs, four-project LOPO

Stage 1 is frozen. This notebook only orchestrates package commands; it does not train Stage 1 or implement Stage 3. Upload the 12 raw ARFF files and optionally authorized manual report exports. Start with CPU. Review retrieval coverage before training. ModernBERT needs a GPU; SBERT can use CPU.

Sync the repository revision containing the new commands to GitHub before using this setup. Do not reuse nonempty experiment output directories. Save caches, prepared data, and outputs as private Kaggle artifacts as appropriate.

In [ ]:
import sys
from pathlib import Path

print("Kernel Python:", sys.executable)

REPO_DIR = Path("/kaggle/working/BugClassiNet")
if not REPO_DIR.exists():
    !git clone https://github.com/Gupta2708/BugClassiNet.git "{REPO_DIR}"
assert (REPO_DIR / "pyproject.toml").is_file()
%cd /kaggle/working/BugClassiNet
!git log -1 --format=%H
%pip install -e . requests beautifulsoup4
!{sys.executable} -m bugclassinet.cli mandelbugs-audit --help

## 1. Locate and audit labels

If files are nested under multiple directories, set RAW_DIR to their common root. Do not choose a dataset by label counts. MANUAL_DIR should point to your authorized CSV/Parquet imports, or remain an empty working folder.

In [ ]:
candidates = list(Path("/kaggle/input").rglob("axis_soap.arff"))
assert len(candidates) == 1, f"Select the intended ARFF dataset explicitly: {candidates}"
RAW_DIR = candidates[0].parent
WORK = Path("/kaggle/working/stage2")
LABEL_DIR = WORK / "labels"
REPORT_DIR = WORK / "reports"
CACHE_DIR = WORK / "raw_response_cache"
PREPARED = WORK / "prepared"
MANUAL_DIR = WORK / "manual_enrichment"  # Change to your mounted manual exports if available.
DATA = PREPARED / "stage2.parquet"
OUTPUT = Path("/kaggle/working/outputs/stage2")
print("ARFF root:", RAW_DIR)
!{sys.executable} -m bugclassinet.cli mandelbugs-audit \
    --raw-dir "{RAW_DIR}" --output-dir "{LABEL_DIR}"

## 2. Enrich reports

Requires permitted internet access. Import the successful first-cloud Linux/AXIS snapshot and your successful local MySQL snapshot with `--prior-reports`. HTTPD uses the documented Apache Bugzilla REST API and an API key from Kaggle Secrets. No secret is printed or saved. 401/403/429 and anti-bot pages are recorded, never bypassed. Set OFFLINE=True only after every legitimate source has been collected. Preserve the cache and reports between Kaggle runs.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

USE_APACHE_API = True
if USE_APACHE_API:
    os.environ["BUGCLASSINET_APACHE_BUGZILLA_API_KEY"] = UserSecretsClient().get_secret(
        "APACHE_BUGZILLA_API_KEY"
    )

# Set these to the two mounted, legitimate report snapshots.
PRIOR_REPORT_PATHS = [
    Path("/kaggle/input/first-cloud-reports/reports.parquet"),
    Path("/kaggle/input/local-mysql-reports/reports.parquet"),
]
assert all(path.is_file() for path in PRIOR_REPORT_PATHS), PRIOR_REPORT_PATHS
PRIOR_ARGS = " ".join(f'--prior-reports "{path}"' for path in PRIOR_REPORT_PATHS)
OFFLINE = False
OFFLINE_FLAG = "--offline" if OFFLINE else ""
!{sys.executable} -m bugclassinet.cli mandelbugs-enrich \
    --labels "{LABEL_DIR / 'labels.parquet'}" \
    --output-dir "{REPORT_DIR}" \
    --cache-dir "{CACHE_DIR}" \
    --manual-dir "{MANUAL_DIR}" \
    --sleep-seconds 1.0 {PRIOR_ARGS} {OFFLINE_FLAG}

## 3. Review coverage and evidence

Inspect failures, project/class coverage, and actual initial/full text. Full comments may contain hindsight information. Retain failed/UNK annotations; never fabricate missing reports. Resolve manual imports and rerun enrichment as needed.

In [ ]:
import json

import pandas as pd

print(json.loads((REPORT_DIR / "retrieval_audit.json").read_text()))
reports = pd.read_parquet(REPORT_DIR / "reports.parquet")
display(pd.crosstab([reports.project, reports.original_class], reports.retrieval_status))
display(reports[["issue_key", "retrieval_status", "retrieval_source", "text_initial"]].head(10))
display(pd.read_csv(REPORT_DIR / "retrieval_failures.csv").head(20))
readiness = json.loads((REPORT_DIR / "stage2_readiness.json").read_text())
display(pd.DataFrame(readiness["projects"]))
print("Candidate LOPO readiness:", readiness["recommended_for_four_project_lopo"])

In [ ]:
COVERAGE_REVIEWED = False  # Change only after reviewing evidence/coverage limitations.
assert COVERAGE_REVIEWED, "Review retrieval coverage and resolve missing evidence first."
!{sys.executable} -m bugclassinet.cli mandelbugs-prepare \
    --labels "{LABEL_DIR / 'labels.parquet'}" \
    --reports "{REPORT_DIR / 'reports.parquet'}" \
    --output-dir "{PREPARED}"
prepared_audit = json.loads((PREPARED / "stage2_dataset_audit.json").read_text())
print(prepared_audit)
assert prepared_audit["readiness"]["recommended_for_four_project_lopo"], (
    "Prepared data is not ready for four-project LOPO"
)

## 4. TF-IDF first (CPU)

The CLI generates all four LOPO folds and a majority baseline. Use a new output name when repeating. Repeat initial/full as separately named evidence conditions, not as hidden tuning against outer projects.

In [ ]:
EVIDENCE_MODE = "initial"
!{sys.executable} -m bugclassinet.cli train-stage2-baseline \
    --data "{DATA}" --model tfidf_svm --evidence-mode "{EVIDENCE_MODE}" \
    --config configs/models/stage2_tfidf_svm.yaml \
    --output-dir "{OUTPUT / ('tfidf_' + EVIDENCE_MODE)}"

## 5. Frozen SBERT + LogisticRegression

Install neural extras in this Stage-2 notebook environment. CPU works; GPU optionally speeds embedding extraction. No encoder fine-tuning occurs. Record the resolved encoder revision and save the cache.

In [ ]:
%pip install -e ".[stage2,transformers]"
!{sys.executable} -m bugclassinet.cli train-stage2-baseline \
    --data "{DATA}" --model sbert_logreg --evidence-mode "{EVIDENCE_MODE}" \
    --config configs/models/stage2_sbert_logreg.yaml \
    --cache-dir "{WORK / 'embedding_cache'}" \
    --output-dir "{OUTPUT / ('sbert_' + EVIDENCE_MODE)}"

## 6. ModernBERT only after baselines (GPU)

Enable a GPU before running this cell. If changing sessions, first save/upload the prepared dataset and point DATA to that unchanged artifact. This command trains 3 seeds × 4 project folds, selecting epochs only on inner validation. It uses one GPU and does not implement cross-run resume. Preserve outputs and do not present partial folds as a complete experiment.

In [ ]:
import torch

RUN_MODERNBERT = False  # Enable after baselines and GPU setup.
assert RUN_MODERNBERT
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
print(torch.cuda.get_device_name(0))
!{sys.executable} -m bugclassinet.cli train-stage2-modernbert \
    --data "{DATA}" --evidence-mode "{EVIDENCE_MODE}" \
    --config configs/models/stage2_modernbert.yaml \
    --output-dir "{OUTPUT / ('modernbert_' + EVIDENCE_MODE)}" \
    --seeds 13 42 97

## 7. Report and preserve

Read summary.json, per_fold_metrics.csv, bootstrap_ci.json, and majority/ alongside all predictions. Macro-F1 is primary, not accuracy alone. Intervals are conditional on the four observed projects. Do not adjust Stage 1, or claim Stage-2 results before completed folds are saved. Use Kaggle Save Version / output download to preserve artifacts; no dataset/model/embedding files belong in Git.